# 01 — Exploratory Data Analysis: M5 Walmart Sales Forecasting

**Goal:** Load, clean, and explore the M5 Walmart dataset to understand sales patterns, 
seasonality, and data structure. Every finding here drives a direct modeling decision 
in subsequent notebooks.

**Dataset:** M5 Forecasting — Accuracy (Kaggle, 2020)  
**Files:** `sales_train_validation.csv`, `calendar.csv`, `sell_prices.csv`

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

## 2. Load Raw Data

The M5 dataset comes in three separate files that each serve a different purpose. 
`sales_train_validation.csv` is in **wide format** — each column is a day — which we 
must melt into long format before any analysis is possible. `calendar.csv` maps day 
IDs to real dates and contains holiday/event/SNAP flags. `sell_prices.csv` gives us 
weekly prices per product per store, which we use to compute revenue.

In [ ]:
# Load all three files
sales_wide = pd.read_csv('../data/raw/sales_train_validation.csv')
calendar   = pd.read_csv('../data/raw/calendar.csv')
prices     = pd.read_csv('../data/raw/sell_prices.csv')

print('sales_train_validation shape:', sales_wide.shape)
print('calendar shape:              ', calendar.shape)
print('sell_prices shape:           ', prices.shape)
print()
print('Sales columns (first 10):', sales_wide.columns[:10].tolist())
print('Sales columns (last 5):  ', sales_wide.columns[-5:].tolist())
print()
print('Calendar columns:', calendar.columns.tolist())
print()
print('Prices columns:  ', prices.columns.tolist())
print()
sales_wide.head(3)

The three source files loaded cleanly and are exactly what we expect:

- **Sales data** is in wide format — 30,490 product-store series as rows, 
  1,913 day columns (`d_1` through `d_1913`) plus 6 identity columns. 
  This gets melted to long format in Section 3, producing ~58M rows.
- **Calendar** has 1,969 rows (1,913 training days + 56 future days for 
  evaluation). It contains two event slots per day (`event_name_1`, 
  `event_name_2`) — we keep only `event_name_1` for now, which captures 
  the primary event. Revisit in feature engineering if needed.
- **Sell prices** has 6.8M rows — weekly price per product per store, 
  joined on `wm_yr_wk` in Section 4.
- **Early day columns show heavy zeros** for HOBBIES items — expected. 
  Low-frequency products have many zero-sale days, which we will quantify 
  in the distribution analysis later.

## 3. Reshape: Wide → Long Format

The sales data has one row per product-store and one column per day (d_1 through d_1913). 
We melt this into long format — one row per product-store-day — so we can join with 
calendar dates and perform time series operations. This is the core reshaping step 
that unlocks all downstream analysis.

In [ ]:
# ID columns to keep as-is
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# Day columns (d_1 through d_1913)
day_cols = [c for c in sales_wide.columns if c.startswith('d_')]

# Melt wide → long
sales_long = sales_wide.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='units_sold'
)

print('Long format shape:', sales_long.shape)
print('Expected:         ', len(sales_wide) * len(day_cols))
print()
sales_long.head(3)

### Output Interpretation

The melt executed perfectly — long format shape matches the expected 
30,490 × 1,913 = **58,327,370 rows exactly**, with zero row loss.

- **8 columns** — the 6 identity columns (`id`, `item_id`, `dept_id`, 
  `cat_id`, `store_id`, `state_id`) plus `d` (day ID) and `units_sold` 
  (our target). Real dates and prices get added in Section 4 via the 
  calendar and price joins.
- **Early rows show `units_sold = 0`** for HOBBIES items on `d_1` — 
  consistent with what we saw in Section 2. These are legitimate zeros, 
  not nulls; no imputation needed.


## 4. Join Calendar

We join the calendar to get real dates, weekdays, event flags, and SNAP indicators 
for each day ID. All string columns are cast to `category` dtype before the merge 
to cut memory usage — without this, the 58M row dataframe would exceed available RAM. 
Prices are **not** joined here; instead they are joined on demand in the specific 
sections that need revenue, so we never hold the full joined frame in memory at once.

In [ ]:
# Downcast immediately to save memory
sales_long['units_sold'] = sales_long['units_sold'].astype('int16')

# Filter calendar to training days only + slim columns
cal_cols = ['d', 'date', 'wm_yr_wk', 'weekday', 'month', 'year',
            'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
calendar_slim = calendar[calendar['d'].isin(sales_long['d'].unique())][cal_cols].copy()

# Convert string IDs to categoricals BEFORE merge to cut memory ~70%
for col in ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd']:
    sales_long[col] = sales_long[col].astype('category')

# Join calendar only (small join — calendar has 1,913 rows)
df = sales_long.merge(calendar_slim, on='d', how='left')
df['date'] = pd.to_datetime(df['date'])

# Convert remaining string cols to categorical
for col in ['weekday', 'event_name_1', 'event_type_1']:
    df[col] = df[col].astype('category')

print('Shape after calendar join:', df.shape)
print()
print(df.dtypes)
print()
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print()
df.head(3)

The calendar join succeeded at **7.20 GB** — manageable without crashing, entirely 
due to categorical encoding cutting string memory ~70%.

- **18 columns** — 6 identity columns, `d`, `units_sold`, `date`, `wm_yr_wk`, 
  `weekday`, `month`, `year`, `event_name_1`, `event_type_1`, and 3 SNAP flags.
- **`d` column stayed as `object`** — that's fine, it's only used as a join key 
  and won't be needed after this point.
- **SNAP flags are `int64`** — we can downcast to `int8` to save memory if needed later.
- **First rows are Jan 29 2011, all zeros, no events** — correct starting point 
  for the dataset.
- **`event_name_1` shows NaN** — expected, most days have no event.

## 5. Data Quality Checks

Before exploring patterns we verify date range, missing values, and the structure 
of the dataset. Note that `sell_price` and `revenue` are not present yet — prices 
are joined on demand in later sections. The key thing to flag here is the 
zero-inflation rate in `units_sold`, which is a defining characteristic of retail 
demand data and directly affects model choice.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# PRINT SUMMARY
# =========================
print('Date range:')
print('  Start:', df['date'].min())
print('  End:  ', df['date'].max())
print()

print('Unique products (item_id):        ', df['item_id'].nunique())
print('Unique stores:                    ', df['store_id'].nunique())
print('Unique product-store series:      ', df['id'].nunique())
print('Unique departments (dept_id):     ', df['dept_id'].nunique())
print('Unique categories (cat_id):       ', df['cat_id'].nunique())
print('Unique states:                    ', df['state_id'].nunique())
print()

print('Missing values:')
print(df[['units_sold', 'event_name_1']].isnull().sum())
print()

print('Units sold stats:')
print(df['units_sold'].describe().round(2))
print()

zero_pct = (df["units_sold"] == 0).mean() * 100
print(f'% days with zero sales: {zero_pct:.1f}%')

# =========================
# DATA FOR CHART
# =========================
non_zero = df[df['units_sold'] > 0]['units_sold']

mean = non_zero.mean()
std = non_zero.std()

lower_1 = mean - std
upper_1 = mean + std

lower_3 = mean - 3*std
upper_3 = mean + 3*std

within_1std = ((non_zero >= lower_1) & (non_zero <= upper_1)).mean() * 100

# =========================
# PLOT (FULL + ZOOMED)
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# --- FULL DISTRIBUTION ---
axes[0].hist(non_zero, bins=50)

axes[0].axvline(mean, linestyle='--', linewidth=2, label='Mean')
axes[0].axvline(lower_1, linestyle=':', linewidth=2, label='±1 Std Dev')
axes[0].axvline(upper_1, linestyle=':', linewidth=2)

axes[0].set_title('Full Distribution (Non-Zero Days)')
axes[0].set_xlabel('Units Sold')
axes[0].set_ylabel('Frequency')

axes[0].text(
    0.97, 0.85,
    f'Zero days: {zero_pct:.1f}%\n{within_1std:.1f}% within ±1 std',
    transform=axes[0].transAxes,
    ha='right',
    va='top',
    bbox=dict(boxstyle='round', facecolor='white')
)

axes[0].legend()

# --- ZOOMED (±3 STD) ---
zoom_data = non_zero[(non_zero >= lower_3) & (non_zero <= upper_3)]

axes[1].hist(zoom_data, bins=50)

axes[1].axvline(mean, linestyle='--', linewidth=2)
axes[1].axvline(lower_1, linestyle=':', linewidth=2)
axes[1].axvline(upper_1, linestyle=':', linewidth=2)

axes[1].set_title('Zoomed View (Within ±3 Std Dev)')
axes[1].set_xlabel('Units Sold')

axes[1].text(
    0.97, 0.85,
    f'Range: [{lower_3:.1f}, {upper_3:.1f}]',
    transform=axes[1].transAxes,
    ha='right',
    va='top',
    bbox=dict(boxstyle='round', facecolor='white')
)

# =========================
# FINALIZE
# =========================
plt.tight_layout()
plt.show()

The dataset is clean and structured exactly as expected:

- **Date range: Jan 29 2011 → Apr 24 2016** — roughly 5.25 years, giving us 
  ~63 months for the aggregate series. This is meaningful history for SARIMA, 
  unlike the synthetic 24-month dataset we worked with before.
- **30,490 product-store series** confirmed (3,049 products × 10 stores).
- **Zero missing values in `units_sold`** — no imputation needed for the target variable.
- **`event_name_1` has 53.6M nulls out of 58.3M rows** — expected. The vast 
  majority of days have no event. NaN here means "normal day", not missing data.
- **68.2% of product-store-days have zero sales** — this is the zero-inflation 
  problem central to retail demand forecasting. The median is 0, mean is 1.13, 
  and max is 763. Most individual series are extremely sparse; the signal lives 
  in the aggregate. This is why we model at the aggregate and department level 
  rather than individual product-store level with SARIMA.
- **std of 3.87 vs mean of 1.13** — highly overdispersed. A standard Gaussian 
  assumption in SARIMA will be strained at the individual series level; 
  aggregate smoothing is essential.

## 6. Create Month Column

We create a `month_dt` column by flooring each date to its month period. 
This is the primary grouping key for all monthly aggregations throughout the EDA. 
Price and revenue columns are joined on demand in the sections that need them.

In [ ]:
df['month_dt'] = df['date'].dt.to_period('M').dt.to_timestamp()

print('Total months:', df['month_dt'].nunique())
print('First month: ', df['month_dt'].min())
print('Last month:  ', df['month_dt'].max())
print()
print('Memory usage:', f'{df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

- **64 months** (Jan 2011 → Apr 2016) — this is our aggregate SARIMA series length. 
  Significantly more history than the 24-month synthetic dataset, giving SARIMA 
  5+ complete seasonal cycles to estimate parameters from rather than just 2.
- **Memory held at 7.67 GB** — stable after adding `month_dt`. We are at the 
  ceiling of what we can hold in one frame; all remaining operations will aggregate 
  before joining prices.

## 7. Category & Department Breakdown

We compute revenue by joining prices at the **weekly aggregate level** rather than 
on the full 58M row frame — this avoids the memory crash from Section 4. We group 
`units_sold` by `store_id`, `item_id`, `wm_yr_wk` first (matching the price join 
key), attach prices, compute revenue, then roll up to category and department. 
This gives identical results to a full row-level join at a fraction of the memory cost.

In [ ]:
# Aggregate units to weekly product-store level (matches price join key)
weekly = (
    df.groupby(['store_id', 'item_id', 'wm_yr_wk', 'cat_id', 'dept_id', 'state_id'], observed=True)['units_sold']
    .sum()
    .reset_index()
)

# Join prices at this aggregated level
weekly = weekly.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
weekly['sell_price'] = weekly['sell_price'].fillna(0).astype('float32')
weekly['revenue'] = (weekly['units_sold'] * weekly['sell_price']).astype('float32')

print('Weekly aggregated shape:', weekly.shape)
print(f'Memory: {weekly.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()

# Category breakdown
cat_totals = (
    weekly.groupby('cat_id', observed=True)['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
cat_totals['pct'] = (cat_totals['revenue'] / cat_totals['revenue'].sum() * 100).round(2)
cat_totals['revenue_fmt'] = cat_totals['revenue'].apply(lambda x: f'${x:,.0f}')
print('Revenue by Category:')
print(cat_totals[['cat_id', 'revenue_fmt', 'pct']].to_string(index=False))
print()

# Department breakdown
dept_totals = (
    weekly.groupby('dept_id', observed=True)['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
dept_totals['pct'] = (dept_totals['revenue'] / dept_totals['revenue'].sum() * 100).round(2)
dept_totals['revenue_fmt'] = dept_totals['revenue'].apply(lambda x: f'${x:,.0f}')
print('Revenue by Department:')
print(dept_totals[['dept_id', 'revenue_fmt', 'pct']].to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(cat_totals['cat_id'][::-1], cat_totals['revenue'][::-1], color='steelblue')
axes[0].set_title('Total Revenue by Category', fontsize=13)
axes[0].set_xlabel('Total Revenue ($)')

axes[1].barh(dept_totals['dept_id'][::-1], dept_totals['revenue'][::-1], color='steelblue')
axes[1].set_title('Total Revenue by Department', fontsize=13)
axes[1].set_xlabel('Total Revenue ($)')

plt.tight_layout()
plt.show()

The weekly aggregation approach worked cleanly — **1.1 GB vs the 7+ GB** the full 
row-level price join would have required.

- **FOODS dominates at 58%** of total platform revenue ($108.9M), nearly double 
  HOUSEHOLD (29.8%) and almost 5x HOBBIES (12.2%). This matters for modeling — 
  FOODS will drive the aggregate series signal almost entirely.
- **FOODS_3 alone is 37.8%** of all revenue — the single largest department by 
  a wide margin. This is likely the produce/perishables department where volume 
  and price are both high.
- **HOBBIES_2 is negligible at 0.63%** ($1.17M) — essentially no signal. 
  Individual series in this department will be extremely sparse and unreliable 
  for standalone SARIMA modeling.
- **Modeling implication:** When we select a representative series for individual 
  SARIMA, we should pick from FOODS_3 or FOODS — not HOBBIES. The representative 
  series needs enough signal to be modelable.

## 8. Revenue Distribution

We examine the distribution of weekly revenue per product-store to assess skewness 
and determine whether a log transformation is needed before modeling. We use the 
weekly aggregated frame rather than daily since that is where revenue lives after 
our memory-efficient price join.

In [ ]:
# Filter to weeks with actual revenue
rev_nonzero = weekly[weekly['revenue'] > 0]['revenue']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rev_nonzero, bins=100, color='steelblue', edgecolor='none')
axes[0].axvline(rev_nonzero.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean:   ${rev_nonzero.mean():.2f}')
axes[0].axvline(rev_nonzero.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: ${rev_nonzero.median():.2f}')
axes[0].set_title('Weekly Revenue per Product-Store — Full Range', fontsize=13)
axes[0].set_xlabel('Revenue ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

p95 = rev_nonzero.quantile(0.95)
axes[1].hist(rev_nonzero[rev_nonzero <= p95], bins=100, color='steelblue', edgecolor='none')
axes[1].set_title(f'Weekly Revenue — Zoomed to 95th Pct (≤ ${p95:.0f})', fontsize=13)
axes[1].set_xlabel('Revenue ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Skewness:                {rev_nonzero.skew():.2f}')
print(f'% weeks with 0 revenue:  {(weekly["revenue"] == 0).mean()*100:.1f}%')
print(f'% revenue under $50:     {(rev_nonzero < 50).mean()*100:.1f}%')
print(f'95th percentile:         ${p95:,.2f}')
print(f'Max weekly revenue:      ${rev_nonzero.max():,.2f}')


- **Skewness of 13.32** — heavily right-skewed but far less extreme than the 
  synthetic dataset (45.37). This is real retail data with genuine variance.
- **40.1% of product-store-weeks have zero revenue** — lower than the 68.2% 
  zero rate on daily units, as expected when aggregating to weekly level. 
  Sparsity improves as we aggregate up.
- **80.8% of non-zero weeks are under $50** and the 95th percentile is just 
  $125 — the vast majority of product-store-weeks are low-value. The max of 
  $7,838 is a genuine outlier, likely a high-volume FOODS_3 product.
- **Log transform decision:** At the aggregate monthly level the CLT will 
  stabilize this considerably — we defer the log transform decision to the 
  SARIMA notebook where we can inspect the monthly series variance directly.

## 9. Monthly Trend — Aggregate Series

We aggregate all product-store weekly revenue to monthly totals, producing the 
primary time series we will model with SARIMA and Prophet. We use the `weekly` 
frame rather than `df` since revenue lives there after our memory-efficient price 
join. We look for visible trend, seasonality, and any structural breaks that would 
affect model choice.

In [ ]:
# Aggregate daily units to monthly, then join prices at monthly-item-store level
# This avoids week boundary artifacts from wm_yr_wk mapping

daily_agg = (
    df.groupby(['store_id', 'item_id', 'month_dt', 'wm_yr_wk'], observed=True)['units_sold']
    .sum()
    .reset_index()
)
daily_agg = daily_agg.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
daily_agg['sell_price'] = daily_agg['sell_price'].fillna(0).astype('float32')
daily_agg['revenue'] = (daily_agg['units_sold'] * daily_agg['sell_price']).astype('float32')

monthly = (
    daily_agg.groupby('month_dt')['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
).sort_values('month_dt').reset_index(drop=True)

monthly['rolling_3'] = monthly['total_revenue'].rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['month_dt'], monthly['total_revenue'], marker='o', linewidth=2, label='Monthly Revenue')
ax.plot(monthly['month_dt'], monthly['rolling_3'], linewidth=2, linestyle='--', color='orange', label='3-Month Rolling Avg')

for _, row in monthly[monthly['month_dt'].dt.month.isin([11, 12])].iterrows():
    label = 'Nov' if row['month_dt'].month == 11 else 'Dec'
    ax.annotate(label, xy=(row['month_dt'], row['total_revenue']),
                xytext=(0, 10), textcoords='offset points', ha='center', fontsize=8, color='red')

ax.set_title('Monthly Total Revenue — All Products & Stores (2011–2016)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Total Revenue ($)')
ax.legend()
plt.tight_layout()
plt.show()

print(monthly[['month_dt', 'total_revenue']].to_string(index=False))

- **Strong upward trend** from ~$2M/month in 2011 to ~$4M/month by early 2016 — 
  roughly doubling over 5 years of consistent Walmart growth.
- **Jan 2011 is $218K** — not a dip, just incomplete data. The dataset starts 
  Jan 29 so only 3 days are captured. We will exclude this point when fitting SARIMA.
- **No single dominant holiday spike** — seasonality is subtle and spread across 
  the year rather than concentrated in one month. The series is trend-driven, 
  not seasonality-driven.
- **Month-to-month variation is smooth** — giving us a clean, reliable series 
  for statistical modeling.

## 10. Seasonal Decomposition — Aggregate Series

We formally separate the monthly revenue series into trend, seasonal, and residual 
components. Unlike synthetic data, we expect a non-flat residual here — genuine 
noise from promotions, weather, and economic conditions that no seasonal model 
can fully explain.

In [ ]:
monthly_indexed = monthly.set_index('month_dt')['total_revenue']

decomp = seasonal_decompose(monthly_indexed, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1],    title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal')
decomp.resid.plot(ax=axes[3],    title='Residual')
for ax in axes:
    ax.set_xlabel('')
plt.suptitle('Seasonal Decomposition — Monthly Aggregate Revenue', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Seasonal component by month:')
print(decomp.seasonal.round(0).to_string())

The decomposition splits the monthly series into 4 separate charts, each isolating 
one component of the signal:

**Observed** — the raw monthly revenue exactly as we plotted in Section 9. 
This is the full signal before any decomposition. You can see the upward trend 
and slight waviness but it is hard to separate what is driving what — that is 
what the next three charts do.

**Trend** — the underlying direction of the business stripped of all seasonality 
and noise. Ours shows a clean, steady climb from ~$2M/month in 2011 to ~$4M/month 
in 2016 with no reversals. This tells us Walmart's revenue was growing consistently 
over this period. The NaNs at the very start and end are normal — additive 
decomposition cannot estimate trend at the edges of the series.

**Seasonal** — the repeating calendar pattern that happens every 12 months 
regardless of trend. August and March are the strongest positive months (+$147K 
and +$128K above baseline), while February and November are the weakest (-$152K 
and -$149K below baseline). Crucially, these effects are small relative to the 
trend — seasonality is a secondary signal here, not the dominant one. This pattern 
will repeat identically in the SARIMA seasonal component.

**Residual** — what is left over after removing trend and seasonality. Everything 
in this chart is variance the model cannot explain structurally — one-off 
promotions, weather events, local store conditions. A well-behaved residual should 
look like random noise with no pattern. If you see spikes or clustering here it 
flags anomalies worth investigating. In our case we expect moderate residual 
noise given this is real retail data across 5+ years.

## 11. Holiday & Event Impact Analysis

We quantify how much named events like the Super Bowl, Thanksgiving, and Christmas 
affect daily revenue compared to a normal day baseline. We compute revenue by 
joining prices at the daily-item-store level, then aggregate to total daily revenue 
per event. These event effects become binary flag features in the XGBoost model.

In [ ]:
# Aggregate daily units at item-store level, join prices, compute revenue
daily_rev = (
    df.groupby(['date', 'store_id', 'item_id', 'wm_yr_wk'], observed=True)['units_sold']
    .sum()
    .reset_index()
)
daily_rev = daily_rev.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
daily_rev['sell_price'] = daily_rev['sell_price'].fillna(0).astype('float32')
daily_rev['revenue'] = (daily_rev['units_sold'] * daily_rev['sell_price']).astype('float32')

# Aggregate to total daily revenue across all stores and items
daily_total = (
    daily_rev.groupby('date')['revenue']
    .sum()
    .reset_index()
)
daily_total.columns = ['date', 'daily_revenue']

# Join event names from calendar separately
event_lookup = calendar[['date', 'event_name_1']].copy()
event_lookup['date'] = pd.to_datetime(event_lookup['date'])
daily_total = daily_total.merge(event_lookup, on='date', how='left')

# Baseline — days with no event
no_event_avg = daily_total[daily_total['event_name_1'].isna()]['daily_revenue'].mean()
print(f'Average daily revenue — no event: ${no_event_avg:,.0f}')
print()

# Event days
event_avg = (
    daily_total[daily_total['event_name_1'].notna()]
    .groupby('event_name_1')['daily_revenue']
    .agg(['mean', 'count'])
    .round(0)
    .sort_values('mean', ascending=False)
    .reset_index()
)
event_avg.columns = ['event_name', 'avg_daily_revenue', 'num_occurrences']
event_avg['vs_baseline'] = ((event_avg['avg_daily_revenue'] / no_event_avg - 1) * 100).round(1).astype(str) + '%'
event_avg['avg_daily_revenue'] = event_avg['avg_daily_revenue'].apply(lambda x: f'${x:,.0f}')

print('Top events by average daily revenue vs baseline:')
print(event_avg.to_string(index=False))

The baseline daily revenue on non-event days is **$98,519**. Event effects are 
modest and tell a very different story than you might expect:

**Events that boost revenue:**
- **Labor Day (+19.6%)** and **Super Bowl (+18.9%)** are the strongest positive 
  drivers — both are high-traffic shopping days where people stock up on food 
  and household supplies before a holiday or gathering.
- **Easter (+16.6%)** and **Orthodox Easter (+13.8%)** show meaningful lifts — 
  consistent with grocery shopping for holiday meals.
- Most other positive events are in the 1–6% range — statistically real but 
  small in absolute terms.

**Events that hurt revenue:**
- **Christmas (-100%)** — stores are closed. The $30 is essentially zero, just 
  residual noise from the price join. This is correct and expected.
- **Thanksgiving (-40.8%)** and **New Year (-23.9%)** — stores either close early 
  or traffic shifts to the day before. The pre-holiday shopping spike shows up 
  in surrounding days, not on the holiday itself.
- **Halloween (-11.3%)**, **Lent (-10–13%)**, **NBA Finals (-5–10%)** — minor 
  negative effects, likely day-of-week confounding more than genuine event impact.

**Modeling implication:** Christmas and Thanksgiving need to be treated as 
**store-closed flags** rather than demand signals. The real signal for these 
holidays lives in the days immediately before them. In XGBoost feature engineering 
we should create lead features — e.g. `days_until_thanksgiving` — rather than 
just a binary event flag on the day itself.

## 12. SNAP Impact Analysis

SNAP benefit days create predictable spending spikes — especially for the FOODS 
category. Each state has a different SNAP schedule, making this a state-specific 
feature. We reuse the `daily_rev` frame from Section 11 and join SNAP flags 
from the calendar to quantify the effect per state.

In [ ]:
# Join SNAP flags from calendar to daily_rev
snap_lookup = calendar[['date', 'snap_CA', 'snap_TX', 'snap_WI']].copy()
snap_lookup['date'] = pd.to_datetime(snap_lookup['date'])

daily_snap = daily_rev.merge(snap_lookup, on='date', how='left')

# Add state_id from df
state_lookup = df[['date', 'store_id', 'state_id']].drop_duplicates()
daily_snap = daily_snap.merge(state_lookup, on=['date', 'store_id'], how='left')

# Add cat_id from df
cat_lookup = df[['item_id', 'store_id', 'cat_id']].drop_duplicates()
daily_snap = daily_snap.merge(cat_lookup, on=['item_id', 'store_id'], how='left')

print('SNAP effect per state — all categories:')
for state in ['CA', 'TX', 'WI']:
    snap_col = f'snap_{state}'
    state_df = daily_snap[daily_snap['state_id'] == state]
    snap_daily = state_df.groupby([snap_col, 'date'])['revenue'].sum().reset_index()
    snap_on  = snap_daily[snap_daily[snap_col] == 1]['revenue'].mean()
    snap_off = snap_daily[snap_daily[snap_col] == 0]['revenue'].mean()
    uplift = (snap_on / snap_off - 1) * 100
    print(f'  {state} — SNAP on: ${snap_on:,.0f} | SNAP off: ${snap_off:,.0f} | Uplift: {uplift:.1f}%')

print()
print('SNAP effect for FOODS category only:')
foods_snap = daily_snap[daily_snap['cat_id'] == 'FOODS']
for state in ['CA', 'TX', 'WI']:
    snap_col = f'snap_{state}'
    state_foods = foods_snap[foods_snap['state_id'] == state]
    snap_daily = state_foods.groupby([snap_col, 'date'])['revenue'].sum().reset_index()
    snap_on  = snap_daily[snap_daily[snap_col] == 1]['revenue'].mean()
    snap_off = snap_daily[snap_daily[snap_col] == 0]['revenue'].mean()
    uplift = (snap_on / snap_off - 1) * 100
    print(f'  {state} — SNAP on: ${snap_on:,.0f} | SNAP off: ${snap_off:,.0f} | Uplift: {uplift:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))

bars = []

for category in plot_df['category'].unique():
    subset = plot_df[plot_df['category'] == category]
    
    bar = ax.bar(
        subset['state'] + '_' + category,
        subset['uplift'],
        label=category
    )
    bars.append(bar)

ax.set_title('SNAP Revenue Uplift by State')
ax.set_ylabel('Uplift (%)')
ax.set_xlabel('State & Category')
ax.axhline(0)

plt.xticks(rotation=45)
plt.legend()

# --- ADD LABELS ON TOP OF BARS ---
for bar_group in bars:
    for rect in bar_group:
        height = rect.get_height()
        ax.text(
            rect.get_x() + rect.get_width()/2,
            height,
            f'{height:.1f}%',
            ha='center',
            va='bottom'
        )

plt.tight_layout()
plt.show()

SNAP benefit days produce a clear, consistent revenue uplift across all three 
states — and the effect is significantly stronger when isolating the FOODS category:

**All categories:**
- **WI has the strongest overall SNAP effect (+20.5%)** — Wisconsin has a smaller 
  store footprint (3 stores) and a higher proportion of SNAP-dependent households, 
  so benefit days move the needle more.
- **TX (+10.8%)** and **CA (+7.2%)** show meaningful but smaller lifts — larger 
  states with more diverse income profiles dilute the effect at the aggregate level.

**FOODS category only:**
- The SNAP effect nearly doubles across all states when isolating FOODS — exactly 
  what we would expect since SNAP benefits can only be spent on food items.
- **WI FOODS is the standout at +32.5%** — nearly a third more revenue on SNAP 
  days vs non-SNAP days. This is a very strong, reliable signal.
- **TX FOODS (+17.2%)** and **CA FOODS (+10.3%)** follow the same pattern.

**Modeling implication:** SNAP flags are among the most valuable features we will 
engineer for XGBoost — they are state-specific, predictable in advance, and 
produce measurable demand lifts especially in FOODS. A single global `is_snap_day` 
flag would be too coarse; we need state-level flags (`snap_CA`, `snap_TX`, 
`snap_WI`) as separate features to capture the different magnitudes per state.

## 13. Department-Level Monthly Trends

We break down monthly revenue by department to assess whether departments move 
together or independently. We reuse the `daily_agg` frame from Section 9 which 
already has revenue computed at the item-store-month level, avoiding another 
expensive price join.

In [ ]:
# Reuse daily_agg from Section 9 — already has revenue and month_dt
# Add dept_id from df
dept_lookup = df[['item_id', 'store_id', 'dept_id']].drop_duplicates()
daily_agg_dept = daily_agg.merge(dept_lookup, on=['item_id', 'store_id'], how='left')

dept_monthly = (
    daily_agg_dept.groupby(['month_dt', 'dept_id'])['revenue']
    .sum()
    .reset_index()
)
dept_pivot = dept_monthly.pivot(index='month_dt', columns='dept_id', values='revenue')

# All departments on one chart
fig, ax = plt.subplots(figsize=(14, 6))
for col in dept_pivot.columns:
    ax.plot(dept_pivot.index, dept_pivot[col], linewidth=1.5, label=col)
ax.set_title('Monthly Revenue by Department', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = dept_pivot.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Between Department Monthly Revenue Series', fontsize=13)
plt.tight_layout()
plt.show()

print(corr.round(2).to_string())

**Monthly trend chart:** All 7 departments show the same upward trend from 2011 
to 2016. FOODS_3 is the dominant department by revenue, sitting well above all 
others throughout the entire period. HOBBIES_2 is barely visible at the bottom — 
consistent with its 0.63% revenue share we saw in Section 7.

**Correlation heatmap:** All departments are positively correlated with each other, 
ranging from 0.64 to 0.96. This tells us departments move together — they are all 
being driven by the same underlying growth trend.

Key correlations to note:
- **HOBBIES_1 and HOUSEHOLD_1 (0.96)** — nearly identical movement, likely driven 
  by the same store traffic patterns.
- **HOBBIES_2 has the weakest correlations across the board (0.64–0.82)** — it is 
  the most independent department, but also the smallest and noisiest so this 
  may reflect sparse signal rather than genuine independence.
- **FOODS_3 and HOBBIES_2 (0.64)** — the weakest pair in the matrix. FOODS_3 is 
  a high-volume staples department while HOBBIES_2 is low-volume discretionary 
  spending — they respond differently to economic conditions.

**Modeling implication:** Correlations are high but not perfect like in purely 
synthetic data — departments have meaningful individual variation on top of the 
shared trend. This means department-level features (`dept_id`) will add genuine 
signal to XGBoost beyond just the aggregate trend. A VAR model across departments 
is not necessary, but encoding department as a feature absolutely is.

## 14. Store-Level Analysis

We assess revenue distribution across the 10 stores and whether stores have 
consistent histories. We reuse `daily_agg` with a store-level groupby — no 
additional price join needed.

In [ ]:
# Store totals — reuse daily_agg which already has revenue and store_id
store_totals = (
    daily_agg.groupby('store_id')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
store_totals['pct'] = (store_totals['revenue'] / store_totals['revenue'].sum() * 100).round(2)
store_totals['revenue_fmt'] = store_totals['revenue'].apply(lambda x: f'${x:,.0f}')

print('Revenue by Store:')
print(store_totals[['store_id', 'revenue_fmt', 'pct']].to_string(index=False))
print()

# Monthly revenue per store
store_monthly = (
    daily_agg.groupby(['month_dt', 'store_id'])['revenue']
    .sum()
    .reset_index()
)
store_pivot = store_monthly.pivot(index='month_dt', columns='store_id', values='revenue')

fig, ax = plt.subplots(figsize=(14, 6))
for col in store_pivot.columns:
    ax.plot(store_pivot.index, store_pivot[col], linewidth=1.5, label=col)
ax.set_title('Monthly Revenue by Store', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()


**Revenue by store:**
- **CA_3 dominates at 17.1%** ($32.1M) — nearly 3x the smallest store (CA_4 
  at 6.5%). This is a significantly larger store, likely a Walmart Supercenter 
  in a high-density California market.
- **CA_1 is second at 12%** — California stores collectively account for ~45% 
  of all revenue despite having 4 of the 10 stores, suggesting CA stores are 
  on average larger and higher volume than TX or WI.
- **TX and WI stores are more evenly distributed** — ranging from 7.9% to 10.9%, 
  without one dominant outlier like CA_3.
- **CA_4 is the smallest at 6.5%** — roughly half of CA_3. These two stores are 
  in the same state but behave very differently, confirming that state alone is 
  not a sufficient feature — store-level encoding is necessary.

**Monthly trend chart:** All 10 stores show the same upward trend from 2011 to 
2016. CA_3 sits visibly above all others throughout the entire period. No store 
shows a structural break or sudden drop, confirming all 10 stores have clean, 
complete histories with no data quality issues.

**Modeling implication:** Store size varies significantly — CA_3 is nearly 3x 
CA_4. Raw revenue targets will reflect store size, not just demand patterns. 
In XGBoost we should encode `store_id` as a categorical feature so the model 
learns store-specific baselines. Alternatively, per-store normalization could 
be applied before modeling if we want the model to focus purely on demand 
patterns rather than volume differences.

## 15. Product-Series Completeness

For individual-level SARIMA modeling we need product-store series with complete 
monthly histories. Products introduced mid-period will have gaps that break 
standard time series assumptions. We use `units_sold` from `df` directly here 
since completeness is about whether sales exist at all — not about revenue.

In [ ]:
total_months = df['month_dt'].nunique()

# Use units_sold from df — no price join needed for completeness check
series_months = (
    df[df['units_sold'] > 0]
    .groupby('id', observed=True)['month_dt']
    .nunique()
    .reset_index()
    .rename(columns={'month_dt': 'active_months'})
)

complete_series = series_months[series_months['active_months'] == total_months]

print(f'Total months in dataset:              {total_months}')
print(f'Total product-store series:           {series_months.shape[0]:,}')
print(f'Series with all {total_months} months:          {len(complete_series):,}')
print(f'Series with < {total_months} months:            {series_months.shape[0] - len(complete_series):,}')
print()
print('Active months distribution:')
print(series_months['active_months'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(series_months['active_months'], bins=total_months, color='steelblue', edgecolor='white')
ax.axvline(total_months, color='red', linestyle='--', linewidth=1.5, label=f'Complete ({total_months} months)')
ax.set_title('Distribution of Active Months per Product-Store Series', fontsize=13)
ax.set_xlabel('Number of Active Months')
ax.set_ylabel('Number of Series')
ax.legend()
plt.tight_layout()
plt.show()


A **series** is one unique product-store combination — e.g. a specific cereal 
tracked at CA_3 over time. There are 30,490 such series in the dataset.

- **Only 2,469 of 30,490 series (8.1%) are complete** across all 64 months. 
  The vast majority of product-store series have gaps — meaning that product 
  was either not yet stocked, discontinued, or simply never sold in certain months.
- **Mean active months is 44.7 out of 64** — the average series is present for 
  about 70% of the dataset's history. This reflects real retail dynamics: products 
  get introduced, rotated, and discontinued regularly.
- **Min of 2 months** — some series have almost no history at all, making them 
  completely unusable for individual SARIMA modeling.
- **The histogram will show a wide spread** — unlike synthetic data where every 
  user had a perfect 24-month history, real retail series vary enormously in length.

**Modeling implication:** Individual-level SARIMA is only viable for the 2,469 
complete series. For XGBoost this is less of a constraint — the model can handle 
varying series lengths since we engineer lag features row by row rather than 
requiring a continuous time index. Our representative series for SARIMA must be 
selected from the 2,469 complete series only.

## 16. Representative Series Selection

We select one product-store series whose monthly revenue is closest to the median 
across all 2,469 complete series. This series becomes our individual-level SARIMA 
proof of concept carried through all modeling notebooks. We build monthly revenue 
by joining prices to the complete series units at the item-store-month level.

In [ ]:
# Get complete series IDs from Section 15
complete_ids = complete_series['id'].tolist()

# Filter df to complete series only, aggregate to item-store-month-week level
complete_df = df[df['id'].isin(complete_ids)]

monthly_rev_complete = (
    complete_df.groupby(['id', 'store_id', 'item_id', 'month_dt', 'wm_yr_wk'], observed=True)['units_sold']
    .sum()
    .reset_index()
)
monthly_rev_complete = monthly_rev_complete.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
monthly_rev_complete['sell_price'] = monthly_rev_complete['sell_price'].fillna(0).astype('float32')
monthly_rev_complete['revenue'] = (monthly_rev_complete['units_sold'] * monthly_rev_complete['sell_price']).astype('float32')

# Roll up to monthly per series
monthly_all_series = (
    monthly_rev_complete.groupby(['id', 'month_dt'], observed=True)['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
monthly_complete = monthly_all_series.copy()

# Find series closest to median total revenue
series_totals = (
    monthly_complete.groupby('id', observed=True)['total_revenue']
    .sum()
    .reset_index()
    .rename(columns={'total_revenue': 'total'})
)
median_total = series_totals['total'].median()
series_totals['dist'] = (series_totals['total'] - median_total).abs()
rep_series = series_totals.loc[series_totals['dist'].idxmin(), 'id']

print(f'Median total revenue across complete series: ${median_total:,.2f}')
print(f'Representative series ID:                    {rep_series}')
print(f'Their total revenue:                         ${series_totals.loc[series_totals["id"] == rep_series, "total"].values[0]:,.2f}')

# Plot representative series
rep_monthly = monthly_complete[monthly_complete['id'] == rep_series].copy().sort_values('month_dt')
rep_monthly['rolling_3'] = rep_monthly['total_revenue'].rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rep_monthly['month_dt'], rep_monthly['total_revenue'], marker='o', linewidth=2, label='Monthly Revenue')
ax.plot(rep_monthly['month_dt'], rep_monthly['rolling_3'], linewidth=2, linestyle='--', color='orange', label='3-Month Rolling Avg')
ax.set_title(f'Monthly Revenue — Series {rep_series}', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.legend()
plt.tight_layout()
plt.show()

# Seasonal decomposition
rep_indexed = rep_monthly.set_index('month_dt')['total_revenue']
rep_decomp = seasonal_decompose(rep_indexed, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
rep_decomp.observed.plot(ax=axes[0], title='Observed')
rep_decomp.trend.plot(ax=axes[1],    title='Trend')
rep_decomp.seasonal.plot(ax=axes[2], title='Seasonal')
rep_decomp.resid.plot(ax=axes[3],    title='Residual')
for ax in axes:
    ax.set_xlabel('')
plt.suptitle(f'Seasonal Decomposition — Series {rep_series}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Output Interpretation

The representative series is **FOODS_3_163_CA_3_validation** — a single food 
product at our highest-revenue store, with a 64-month total of $7,703 (~$120/month 
average). The decomposition reveals four things:

**Observed:** Noisy month-to-month movement ranging from ~$40 to ~$200. This is 
completely normal for a single product — one promotional event, a brief stockout, 
or a price change can easily double or halve a month's revenue. This is why 
individual series are harder to model than aggregates.

**Trend:** Starts around $90/month in early 2011, rises to ~$135 by mid-2012, 
dips slightly through 2013–2014, then climbs again to ~$160 by 2016. The product 
is growing over time but not linearly — it has its own lifecycle curve independent 
of the overall store trend.

**Seasonal:** Clear repeating pattern every 12 months with swings of roughly 
±$40 around the baseline. There is a consistent trough around November–December 
each year (dropping ~$40 below baseline) and peaks in spring and summer. This 
confirms a 12-month seasonal period is appropriate for SARIMA on this series.

**Residual:** Ranges from about -$60 to +$60 with no obvious pattern — genuine 
random noise from promotions, local events, and stockouts. The residual is larger 
relative to the signal than the aggregate series, which means individual-level 
SARIMA confidence intervals will be wide. This is expected and should be 
communicated clearly in the modeling notebook.

**Modeling implication:** The series is modelable — it has clear trend and 
seasonality — but individual-level forecasts will carry more uncertainty than 
aggregate forecasts. SARIMA on this series is a proof of concept, not a 
production-grade individual forecast.

## 17. Price Volatility Analysis

Unlike static pricing in synthetic datasets, M5 products have prices that change 
over time. Price volatility is a real demand driver — a price drop often causes 
a sales spike — and will be an important feature in XGBoost. We join prices 
directly from the `prices` dataframe at the department level, keeping the 
operation lightweight.

In [ ]:
# Join dept_id to prices via item_id lookup
dept_item_lookup = df[['item_id', 'dept_id']].drop_duplicates()
prices_dept = prices.merge(dept_item_lookup, on='item_id', how='left')

# Price volatility per department
price_stats = (
    prices_dept[prices_dept['sell_price'] > 0]
    .groupby('dept_id')['sell_price']
    .agg(['mean', 'std', 'min', 'max'])
    .round(2)
    .reset_index()
)
price_stats['cv'] = (price_stats['std'] / price_stats['mean'] * 100).round(1)
price_stats.columns = ['dept_id', 'mean_price', 'std_price', 'min_price', 'max_price', 'cv_%']

print('Price statistics by department (cv% = coefficient of variation):')
print(price_stats.to_string(index=False))
print()

# Price trend over time for top revenue department (FOODS_3)
top_dept = dept_totals.iloc[0]['dept_id']

# Join date to prices via calendar wm_yr_wk lookup
wk_to_date = calendar[['wm_yr_wk', 'date']].copy()
wk_to_date['date'] = pd.to_datetime(wk_to_date['date'])
wk_to_date['month_dt'] = wk_to_date['date'].dt.to_period('M').dt.to_timestamp()
wk_to_month = wk_to_date.groupby('wm_yr_wk')['month_dt'].first().reset_index()

prices_dept_dated = prices_dept.merge(wk_to_month, on='wm_yr_wk', how='left')

price_trend = (
    prices_dept_dated[
        (prices_dept_dated['dept_id'] == top_dept) &
        (prices_dept_dated['sell_price'] > 0)
    ]
    .groupby('month_dt')['sell_price']
    .mean()
    .reset_index()
).sort_values('month_dt')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(price_trend['month_dt'], price_trend['sell_price'], linewidth=2, color='steelblue')
ax.set_title(f'Average Monthly Price — {top_dept}', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Average Price ($)')
plt.tight_layout()
plt.show()


The coefficient of variation (cv%) measures price volatility relative to the 
mean — a higher cv% means prices fluctuate more wildly around their average.

**Key findings by department:**
- **HOBBIES_1 has the highest price volatility (83.1%)** with a mean of $6.23 
  but a range of $0.01 to $30.98. Hobby products span a huge price spectrum — 
  from small accessories to larger items — and are frequently discounted.
- **HOUSEHOLD_2 has the widest absolute price range** ($0.05 to $107.32) — 
  likely contains some large household items that sit alongside low-cost 
  consumables in the same department.
- **FOODS departments are the most stable (cv% 62–64%)** — grocery prices 
  change but within a tighter band. FOODS_3 has the lowest mean price at $2.84, 
  consistent with it being a high-volume staples department where items are 
  cheap but sell in large quantities.
- **All departments show cv% above 50%** — price volatility is a universal 
  characteristic of this dataset, not isolated to one category. This confirms 
  that `sell_price` and price-change features will add meaningful signal in 
  XGBoost across all departments, not just for high-price items.

**Price trend chart:** The average FOODS_3 price over time will show whether 
Walmart was adjusting prices upward, downward, or holding steady across 2011–2016. 
Any visible trend here would suggest a price feature is needed beyond just the 
current week's price — a rolling price average or price-change percentage would 
capture this drift.

**Modeling implication:** Given cv% above 60% across all departments, we should 
engineer at least three price features in XGBoost: `sell_price` (current), 
`price_lag_1wk` (last week), and `price_change_pct` (week-over-week change). 
These are confirmed as high-value features.

## 18. Price Elasticity Analysis

Price changes are among the strongest demand drivers in retail — a discount
typically triggers a sales spike while a price increase suppresses demand.
We analyze price elasticity at three levels of granularity to understand
where the signal lives. Department-level aggregation is tested first, then
scoped to one store, then to one product. This three-level analysis directly
determines how price features must be engineered in XGBoost — at what
granularity they carry meaningful predictive signal.

A negative slope in the scatter plots is the expected signal — price drops
(negative x) should produce demand increases (positive y), and price
increases (positive x) should produce demand decreases (negative y).
We split drops and increases separately since retail demand responds
asymmetrically to price changes in each direction.

In [ ]:
wk_to_date = calendar[['wm_yr_wk', 'date']].copy()
wk_to_date['date'] = pd.to_datetime(wk_to_date['date'])
wk_to_month = wk_to_date.groupby('wm_yr_wk')['date'].min().reset_index()

# ── LEVEL 1: Department-level — all products, all stores ──────────────────

elasticity_df = (
    weekly[weekly['sell_price'] > 0]
    .merge(wk_to_month, on='wm_yr_wk', how='left')
    .sort_values(['store_id', 'item_id', 'date'])
)
elasticity_df['price_chg_pct'] = (
    elasticity_df.groupby(['store_id', 'item_id'], observed=True)['sell_price']
    .pct_change() * 100
)
elasticity_df['units_chg_pct'] = (
    elasticity_df.groupby(['store_id', 'item_id'], observed=True)['units_sold']
    .pct_change() * 100
)
elasticity_clean = elasticity_df.dropna(subset=['price_chg_pct', 'units_chg_pct'])
elasticity_clean = elasticity_clean[
    (elasticity_clean['price_chg_pct'].abs() < 50) &
    (elasticity_clean['units_chg_pct'].abs() < 500) &
    (elasticity_clean['price_chg_pct'] != 0)
]

dept_drops     = elasticity_clean[elasticity_clean['price_chg_pct'] < 0]
dept_increases = elasticity_clean[elasticity_clean['price_chg_pct'] > 0]

print('=' * 62)
print('LEVEL 1 — Department Level (all products, all stores)')
print('=' * 62)
dept_elasticity = []
for dept in sorted(elasticity_clean['dept_id'].unique()):
    for direction, subset in [('drop', dept_drops[dept_drops['dept_id'] == dept]),
                               ('increase', dept_increases[dept_increases['dept_id'] == dept])]:
        if len(subset) < 30:
            continue
        coeffs = np.polyfit(subset['price_chg_pct'], subset['units_chg_pct'], 1)
        corr   = subset['price_chg_pct'].corr(subset['units_chg_pct'])
        dept_elasticity.append({
            'dept_id': dept, 'direction': direction,
            'elasticity': round(coeffs[0], 3),
            'correlation': round(corr, 3), 'n_obs': len(subset)
        })
elast_df = pd.DataFrame(dept_elasticity)
print(f"{'Dept':<15} {'Direction':<12} {'Elasticity':>12} {'Correlation':>12} {'N obs':>8}")
print('-' * 62)
for _, row in elast_df.iterrows():
    print(f"{row['dept_id']:<15} {row['direction']:<12} {row['elasticity']:>12.3f} "
          f"{row['correlation']:>12.3f} {row['n_obs']:>8,}")

# ── LEVEL 2: FOODS_3 @ CA_3 — one department, one store ──────────────────

print()
print('=' * 62)
print('LEVEL 2 — FOODS_3 @ CA_3 (one department, one store)')
print('=' * 62)

foods3_ca3 = prices[
    (prices['store_id'] == 'CA_3') &
    (prices['item_id'].str.startswith('FOODS_3'))
].sort_values(['item_id', 'wm_yr_wk']).copy()
foods3_ca3['price_chg_pct'] = (
    foods3_ca3.groupby('item_id')['sell_price'].pct_change() * 100
)
units_f3 = (
    df[(df['store_id'] == 'CA_3') & (df['item_id'].str.startswith('FOODS_3'))]
    .groupby(['item_id', 'wm_yr_wk'], observed=True)['units_sold']
    .sum().reset_index()
)
units_f3['units_chg_pct'] = (
    units_f3.groupby('item_id', observed=True)['units_sold'].pct_change() * 100
)
foods3_clean = foods3_ca3.merge(units_f3, on=['item_id', 'wm_yr_wk'], how='left')
foods3_clean = foods3_clean[
    (foods3_clean['price_chg_pct'].abs() > 0) &
    (foods3_clean['price_chg_pct'].abs() < 50) &
    (foods3_clean['units_chg_pct'].abs() < 500)
].dropna()

f3_drops     = foods3_clean[foods3_clean['price_chg_pct'] < 0]
f3_increases = foods3_clean[foods3_clean['price_chg_pct'] > 0]

for direction, subset in [('Price drops', f3_drops), ('Price increases', f3_increases)]:
    coeffs = np.polyfit(subset['price_chg_pct'], subset['units_chg_pct'], 1)
    corr   = subset['price_chg_pct'].corr(subset['units_chg_pct'])
    print(f'{direction:<20} n={len(subset):,}  elasticity={coeffs[0]:.3f}  r={corr:.3f}')

# ── LEVEL 3: FOODS_3_383 @ CA_3 — single product ─────────────────────────

print()
print('=' * 62)
print('LEVEL 3 — FOODS_3_383 @ CA_3 (single product)')
print('=' * 62)

top_item  = 'FOODS_3_383'
top_store = 'CA_3'

top_weekly = (
    df[(df['item_id'] == top_item) & (df['store_id'] == top_store)]
    .groupby('wm_yr_wk', observed=True)['units_sold'].sum().reset_index()
)
top_weekly = top_weekly.merge(
    prices[(prices['item_id'] == top_item) & (prices['store_id'] == top_store)],
    on='wm_yr_wk', how='left'
)
top_weekly = top_weekly.merge(wk_to_month, on='wm_yr_wk', how='left')
top_weekly = top_weekly.sort_values('date').dropna(subset=['sell_price'])
top_weekly['price_chg_pct'] = top_weekly['sell_price'].pct_change() * 100
top_weekly['units_chg_pct'] = top_weekly['units_sold'].pct_change() * 100

top_changes   = top_weekly[
    (top_weekly['price_chg_pct'].abs() > 0) &
    (top_weekly['price_chg_pct'].abs() < 50) &
    (top_weekly['units_chg_pct'].abs() < 500)
].dropna()
top_drops     = top_changes[top_changes['price_chg_pct'] < 0]
top_increases = top_changes[top_changes['price_chg_pct'] > 0]

for direction, subset in [('Price drops', top_drops), ('Price increases', top_increases)]:
    if len(subset) < 2:
        print(f'{direction:<20} n={len(subset)} — insufficient data')
        continue
    coeffs = np.polyfit(subset['price_chg_pct'], subset['units_chg_pct'], 1)
    corr   = subset['price_chg_pct'].corr(subset['units_chg_pct'])
    print(f'{direction:<20} n={len(subset):,}  elasticity={coeffs[0]:.3f}  r={corr:.3f}')

# ── PLOTS ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# Level 1 — Department drops
dept_drop_all     = dept_drops
dept_increase_all = dept_increases
c_drop = np.polyfit(dept_drop_all['price_chg_pct'],     dept_drop_all['units_chg_pct'],     1)
c_inc  = np.polyfit(dept_increase_all['price_chg_pct'], dept_increase_all['units_chg_pct'], 1)
r_drop = dept_drop_all['price_chg_pct'].corr(dept_drop_all['units_chg_pct'])
r_inc  = dept_increase_all['price_chg_pct'].corr(dept_increase_all['units_chg_pct'])

axes[0][0].scatter(dept_drop_all['price_chg_pct'], dept_drop_all['units_chg_pct'],
                   alpha=0.05, s=5, color='green')
x = np.linspace(dept_drop_all['price_chg_pct'].min(), dept_drop_all['price_chg_pct'].max(), 100)
axes[0][0].plot(x, c_drop[0]*x + c_drop[1], color='darkgreen', linewidth=2)
axes[0][0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0][0].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[0][0].set_title(f'Level 1: All Depts — Price Drops\n(n={len(dept_drop_all):,}, r={r_drop:.3f})', fontsize=11)
axes[0][0].set_xlabel('Price Change %')
axes[0][0].set_ylabel('Demand Change %')

axes[0][1].scatter(dept_increase_all['price_chg_pct'], dept_increase_all['units_chg_pct'],
                   alpha=0.05, s=5, color='red')
x = np.linspace(dept_increase_all['price_chg_pct'].min(), dept_increase_all['price_chg_pct'].max(), 100)
axes[0][1].plot(x, c_inc[0]*x + c_inc[1], color='darkred', linewidth=2)
axes[0][1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0][1].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[0][1].set_title(f'Level 1: All Depts — Price Increases\n(n={len(dept_increase_all):,}, r={r_inc:.3f})', fontsize=11)
axes[0][1].set_xlabel('Price Change %')
axes[0][1].set_ylabel('Demand Change %')

# Level 2 — FOODS_3 @ CA_3
c_f3d = np.polyfit(f3_drops['price_chg_pct'],     f3_drops['units_chg_pct'],     1)
c_f3i = np.polyfit(f3_increases['price_chg_pct'], f3_increases['units_chg_pct'], 1)
r_f3d = f3_drops['price_chg_pct'].corr(f3_drops['units_chg_pct'])
r_f3i = f3_increases['price_chg_pct'].corr(f3_increases['units_chg_pct'])

axes[1][0].scatter(f3_drops['price_chg_pct'], f3_drops['units_chg_pct'],
                   alpha=0.2, s=10, color='green')
x = np.linspace(f3_drops['price_chg_pct'].min(), f3_drops['price_chg_pct'].max(), 100)
axes[1][0].plot(x, c_f3d[0]*x + c_f3d[1], color='darkgreen', linewidth=2)
axes[1][0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1][0].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[1][0].set_title(f'Level 2: FOODS_3 @ CA_3 — Price Drops\n(n={len(f3_drops):,}, r={r_f3d:.3f})', fontsize=11)
axes[1][0].set_xlabel('Price Change %')
axes[1][0].set_ylabel('Demand Change %')

axes[1][1].scatter(f3_increases['price_chg_pct'], f3_increases['units_chg_pct'],
                   alpha=0.2, s=10, color='red')
x = np.linspace(f3_increases['price_chg_pct'].min(), f3_increases['price_chg_pct'].max(), 100)
axes[1][1].plot(x, c_f3i[0]*x + c_f3i[1], color='darkred', linewidth=2)
axes[1][1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1][1].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[1][1].set_title(f'Level 2: FOODS_3 @ CA_3 — Price Increases\n(n={len(f3_increases):,}, r={r_f3i:.3f})', fontsize=11)
axes[1][1].set_xlabel('Price Change %')
axes[1][1].set_ylabel('Demand Change %')

# Level 3 — Single product scatter: drops and increases
if len(top_drops) >= 2:
    c_t3d = np.polyfit(top_drops['price_chg_pct'], top_drops['units_chg_pct'], 1)
    r_t3d = top_drops['price_chg_pct'].corr(top_drops['units_chg_pct'])
    axes[2][0].scatter(top_drops['price_chg_pct'], top_drops['units_chg_pct'],
                       alpha=0.7, s=40, color='green')
    x = np.linspace(top_drops['price_chg_pct'].min(), top_drops['price_chg_pct'].max(), 100)
    axes[2][0].plot(x, c_t3d[0]*x + c_t3d[1], color='darkgreen', linewidth=2)
    axes[2][0].set_title(f'Level 3: {top_item} @ {top_store} — Price Drops\n(n={len(top_drops)}, r={r_t3d:.3f})', fontsize=11)
else:
    axes[2][0].set_title(f'Level 3: {top_item} — insufficient drop data', fontsize=11)
axes[2][0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[2][0].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[2][0].set_xlabel('Price Change %')
axes[2][0].set_ylabel('Demand Change %')

if len(top_increases) >= 2:
    c_t3i = np.polyfit(top_increases['price_chg_pct'], top_increases['units_chg_pct'], 1)
    r_t3i = top_increases['price_chg_pct'].corr(top_increases['units_chg_pct'])
    axes[2][1].scatter(top_increases['price_chg_pct'], top_increases['units_chg_pct'],
                       alpha=0.7, s=40, color='red')
    x = np.linspace(top_increases['price_chg_pct'].min(), top_increases['price_chg_pct'].max(), 100)
    axes[2][1].plot(x, c_t3i[0]*x + c_t3i[1], color='darkred', linewidth=2)
    axes[2][1].set_title(f'Level 3: {top_item} @ {top_store} — Price Increases\n(n={len(top_increases)}, r={r_t3i:.3f})', fontsize=11)
else:
    axes[2][1].set_title(f'Level 3: {top_item} — insufficient increase data', fontsize=11)
axes[2][1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[2][1].axvline(0, color='black', linewidth=0.5, linestyle='--')
axes[2][1].set_xlabel('Price Change %')
axes[2][1].set_ylabel('Demand Change %')

plt.suptitle('Price Elasticity Analysis — Three Levels of Granularity', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Price Elasticity Results — Three Levels of Granularity

**How to read the scatter plots:**
- X axis: week-over-week price change %. Negative = price fell, positive = price rose.
- Y axis: week-over-week demand change %. Negative = demand fell, positive = demand rose.
- The expected rational signal is a **negative slope** — price drops in the top-left
  quadrant (price fell, demand rose) and price increases in the bottom-right
  (price rose, demand fell).
- **Why slopes are positive instead:** Walmart discounts products that are already
  selling poorly — slow movers get marked down to clear inventory. Price drops and
  low demand are both caused by poor product performance, not each other. This
  confounding dominates at aggregate levels and produces a spurious positive slope.
  The true demand response is buried in seasonal, event, and SNAP noise.

**Level 1 — All departments, all stores (r = 0.003–0.129):**
Near-zero correlations throughout. A price change on one product gets averaged
across thousands of other product-store combinations with no price change,
diluting the signal entirely. Three departments show negative elasticity on
drops (HOBBIES_2, HOUSEHOLD_1, HOUSEHOLD_2) — a statistical artifact of
aggregation, not genuine consumer behavior.

**Level 2 — FOODS_3 @ CA_3 (r = 0.158–0.185):**
Scoping to one store improves the signal modestly. Elasticity rises to
1.6–1.7, meaning a 10% price change produces roughly 16% demand response
across FOODS_3 products at CA_3. Drops and increases show similar elasticity
at this level — product-specific asymmetry is still being averaged away.

**Level 3 — FOODS_3_383 @ CA_3 (drops r = 0.553, increases r = 0.180):**
At the individual product-store level the expected asymmetry finally appears.
Price drops produce a strong, consistent demand response (r=0.553, elasticity=7.7
— a 10% drop is associated with ~77% demand increase). Price increases show a
much weaker response (r=0.180). This is **asymmetric price elasticity** — customers
stock up aggressively on discounts but do not proportionally reduce purchases
when prices rise. Note n=16 and n=13 are small samples; XGBoost learns these
patterns across thousands of products simultaneously, making estimates far
more robust.

**Feature engineering implications for notebook 4:**

| Feature | Decision | Reasoning |
|---|---|---|
| `sell_price` | ✓ Include | Current price level carries signal at product-store level |
| `price_change_pct` (signed) | ✓ Include | Sign must be preserved — drops and increases behave differently |
| `price_drop_pct` | ✓ Include | Asymmetric elasticity confirmed — drops deserve a separate feature |
| `price_increase_pct` | ✓ Include | Weaker but present signal |
| Department-average price change | ✗ Exclude | r < 0.13 at department level — no predictive value |

All price features must be computed at the `item_id` + `store_id` level.
Any aggregation above that level destroys the signal as demonstrated above.

## 19. Save Processed Files

We save four files that feed directly into the modeling notebooks. The long-format 
transaction table is the foundation for XGBoost feature engineering. The monthly 
series files are inputs for SARIMA and Prophet.

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# 1. Aggregate monthly series — SARIMA/Prophet
monthly[['month_dt', 'total_revenue']].to_csv(
    '../data/processed/monthly_aggregate.csv', index=False)
print(f'monthly_aggregate.csv saved:              {monthly.shape[0]} rows')

# 2. Representative series — individual SARIMA/Prophet
rep_monthly[['month_dt', 'total_revenue']].to_csv(
    f'../data/processed/monthly_series_{rep_series}.csv', index=False)
print(f'monthly_series_{rep_series}.csv saved:    {rep_monthly.shape[0]} rows')

# 3. Complete series IDs — filter reference for notebook 3
pd.DataFrame({'id': complete_ids}).to_csv(
    '../data/processed/complete_series_ids.csv', index=False)
print(f'complete_series_ids.csv saved:            {len(complete_ids):,} rows')

print()
print(f'Representative series ID: {rep_series}')
print('Hardcode this value in all modeling notebooks.')

## 20. Stationarity Testing — ADF Test

SARIMA requires each series to be **stationary** — meaning its mean and variance
are stable over time with no persistent trend. The Augmented Dickey-Fuller (ADF)
test formalizes this check: a p-value below 0.05 means we can treat the series
as stationary; above 0.05 means a trend is present and differencing is required.
We test the raw series, then after first differencing and seasonal differencing
(lag-12), to determine the correct `d` and `D` parameters for each SARIMA model.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, label):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"{'─'*50}")
    print(f"Series:      {label}")
    print(f"ADF stat:    {result[0]:.4f}")
    print(f"p-value:     {result[1]:.4f}")
    print(f"Lags used:   {result[2]}")
    print(f"Critical values:")
    for k, v in result[4].items():
        print(f"  {k}: {v:.4f}")
    conclusion = "STATIONARY ✓" if result[1] < 0.05 else "NON-STATIONARY ✗"
    print(f"Conclusion:  {conclusion}")
    print()

# Load the saved processed files (avoids rebuilding from 58M row frame)
monthly_agg    = pd.read_csv('../data/processed/monthly_aggregate.csv', parse_dates=['month_dt'])
monthly_rep    = pd.read_csv('../data/processed/monthly_series_FOODS_3_163_CA_3_validation.csv', parse_dates=['month_dt'])

agg_raw  = monthly_agg.set_index('month_dt')['total_revenue']
rep_raw  = monthly_rep.set_index('month_dt')['total_revenue']

# --- Raw series ---
print("=== RAW SERIES ===\n")
adf_report(agg_raw, "Aggregate — raw")
adf_report(rep_raw, "Representative (FOODS_3_163_CA_3) — raw")

# --- First difference ---
print("=== FIRST DIFFERENCE ===\n")
adf_report(agg_raw.diff(), "Aggregate — first diff")
adf_report(rep_raw.diff(), "Representative (FOODS_3_163_CA_3) — first diff")

# --- Seasonal difference (lag-12) ---
print("=== SEASONAL DIFFERENCE (lag-12) ===\n")
adf_report(agg_raw.diff(12), "Aggregate — seasonal diff")
adf_report(rep_raw.diff(12), "Representative (FOODS_3_163_CA_3) — seasonal diff")

### Stationarity Results & SARIMA Parameter Decisions

**What the test measures:** A p-value below 0.05 means the series has no
persistent trend and is safe to model directly. Above 0.05 means a trend
is present — we remove it by differencing before fitting SARIMA.

**Aggregate series:**
- Raw: p = 0.318 — **non-stationary**. The upward trend from 2011–2016
  violates SARIMA's core assumption. The series mean is not stable.
- First difference: p = 0.059 — still borderline non-stationary. Subtracting
  consecutive months removes the linear trend but leaves residual structure.
- Seasonal difference (lag-12): p ≈ 0.000 — **stationary**. Subtracting
  each month from the same month one year prior fully removes the drift.
  The seasonal cycle is the dominant structure in this series.
- **Decision: d=0, D=1** for the aggregate SARIMA model.

**Representative series (FOODS_3_163_CA_3):**
- Raw: p = 0.0004 — already **stationary**. This individual product series
  does not have a persistent trend — it fluctuates around a stable mean
  throughout the full 64-month window.
- Remains stationary under all differencing forms.
- **Decision: d=0, D=0** as the baseline, but D=1 will be included in the
  parameter grid since seasonal decomposition confirmed an annual cycle.

| Series | d | D | Reasoning |
|---|---|---|---|
| Aggregate | 0 | 1 | Seasonal differencing resolves non-stationarity |
| Representative | 0 | 0 | Already stationary; D=1 tested in grid search |

## 21. ACF and PACF Analysis

The Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF)
plots tell us the correct AR and MA lag orders for SARIMA — the `p`, `q`, `P`,
and `Q` parameters. ACF measures how correlated the series is with its own past
values at each lag. PACF measures the same but strips out indirect effects of
intermediate lags, isolating only the direct relationship at each distance.
We plot both series after applying their confirmed differencing transformations
from Section 20.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf

agg_diff = agg_raw.diff(12).dropna()
rep_diff = rep_raw.dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(agg_diff,  lags=24, ax=axes[0][0], title='ACF — Aggregate (seasonal diff)')
plot_pacf(agg_diff, lags=24, ax=axes[0][1], title='PACF — Aggregate (seasonal diff)')
plot_acf(rep_diff,  lags=24, ax=axes[1][0], title='ACF — Representative (raw)')
plot_pacf(rep_diff, lags=24, ax=axes[1][1], title='PACF — Representative (raw)')

for ax in axes.flat:
    ax.set_xlabel('Lag (months)')
    ax.axvline(x=12, color='red', linestyle='--', linewidth=1, alpha=0.5, label='lag-12')
    ax.legend(fontsize=8)

plt.suptitle('ACF and PACF — Aggregate and Representative Series', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

def acf_pacf_table(series, label, nlags=24):
    n = len(series)
    conf = 1.96 / np.sqrt(n)
    acf_vals  = acf(series,  nlags=nlags, fft=True)[1:]
    pacf_vals = pacf(series, nlags=nlags)[1:]

    print(f"\n{'═'*60}")
    print(f"Series: {label}  |  n={n}  |  95% confidence bound: ±{conf:.3f}")
    print(f"{'═'*60}")
    print(f"{'Lag':>4}  {'ACF':>8}  {'ACF sig?':>10}  {'PACF':>8}  {'PACF sig?':>10}")
    print(f"{'─'*60}")
    for i, (a, p) in enumerate(zip(acf_vals, pacf_vals), start=1):
        a_sig = '✓' if abs(a) > conf else ''
        p_sig = '✓' if abs(p) > conf else ''
        print(f"{i:>4}  {a:>8.3f}  {a_sig:>10}  {p:>8.3f}  {p_sig:>10}")

acf_pacf_table(agg_diff, "Aggregate (seasonal diff)")
acf_pacf_table(rep_diff, "Representative — FOODS_3_163_CA_3 (raw)")

### ACF/PACF Results & SARIMA Parameter Decisions

**How to read these plots:** Each bar is one lag (one month back in time).
A bar that crosses the shaded confidence band is significant — that lag contains
real predictive information. PACF spikes tell us the AR order (`p`, `P`);
ACF spikes tell us the MA order (`q`, `Q`). We look at early lags (1–3) for
non-seasonal parameters and specifically at lag-12 for seasonal parameters.

**Aggregate series (after seasonal differencing):**
- PACF: one significant spike at lag-1, then nothing → **p = 1**
- ACF: significant at lags 1–5 but decays gradually rather than cutting off
  cleanly → **q = 0** (gradual decay is the signature of AR carry-through,
  not a separate MA process)
- Lag-12: neither ACF nor PACF significant — seasonal differencing fully
  removed the annual structure → **P = 0, Q = 0**
- **Candidate model: SARIMA(1,0,0)(0,1,0)[12]**

**Representative series (no differencing):**
- PACF: one significant spike at lag-1 only → **p = 1**
- ACF: significant at lags 1–2 then cuts off → **q = 1**
- Lag-12: neither ACF nor PACF significant → **P = 0, Q = 0**
- Isolated PACF spikes at lags 23–24 are noise — with n=64, ~5% of lags
  will cross the confidence bound by chance even in a white noise series
- **Candidate model: SARIMA(1,0,1)(0,0,0)[12]**

**Grid search ranges for notebook 2:**

| Series | p | d | q | P | D | Q |
|---|---|---|---|---|---|---|
| Aggregate | 0–2 | 0 | 0–1 | 0–1 | 1 | 0–1 |
| Representative | 0–2 | 0 | 0–1 | 0 | 0 | 0 |

AIC selects the final order within these ranges — the candidates above
are the expected winners but the grid search confirms it.

## 22. Units Sold Distribution & Zero-Streak Analysis

We know from Section 5 that 68.2% of **product-store-days** have zero sales —
meaning one specific product at one specific store sold nothing on that day.
Two things remain unquantified: the shape of the non-zero demand distribution
at that same granularity (which affects whether a Gaussian SARIMA assumption
is reasonable), and whether zeros cluster into long consecutive streaks
(structural gaps where a product wasn't stocked) or are scattered randomly
(intermittent demand). These are meaningfully different problems — clustered
zeros indicate product lifecycle events while scattered zeros are normal sparse
demand. We check both using `units_sold` directly from `df`.

In [ ]:
# --- Non-zero units sold distribution ---
units_nonzero = df[df['units_sold'] > 0]['units_sold']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Full range
axes[0].hist(units_nonzero, bins=100, color='steelblue', edgecolor='none')
axes[0].axvline(units_nonzero.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean:   {units_nonzero.mean():.1f}')
axes[0].axvline(units_nonzero.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {units_nonzero.median():.1f}')
axes[0].set_title('Units Sold — Full Range (non-zero)', fontsize=12)
axes[0].set_xlabel('Units Sold')
axes[0].set_ylabel('Count')
axes[0].legend()

# Zoomed to 95th percentile
p95 = units_nonzero.quantile(0.95)
axes[1].hist(units_nonzero[units_nonzero <= p95], bins=100, color='steelblue', edgecolor='none')
axes[1].set_title(f'Units Sold — Zoomed to 95th Pct (≤{p95:.0f})', fontsize=12)
axes[1].set_xlabel('Units Sold')
axes[1].set_ylabel('Count')

# By category
for cat, grp in df[df['units_sold'] > 0].groupby('cat_id', observed=True)['units_sold']:
    axes[2].hist(grp[grp <= p95], bins=60, alpha=0.6, edgecolor='none', label=cat)
axes[2].set_title('Units Sold by Category (≤ 95th Pct)', fontsize=12)
axes[2].set_xlabel('Units Sold')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.suptitle('Units Sold Distribution — Non-Zero Days', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Units sold (non-zero) stats:')
print(units_nonzero.describe().round(2))
print(f'\nSkewness:        {units_nonzero.skew():.2f}')
print(f'% days = 1 unit: {(df["units_sold"] == 1).mean()*100:.1f}%')
print(f'% days ≤ 3 units:{(df["units_sold"] <= 3).mean()*100:.1f}%')
print(f'95th percentile: {p95:.0f} units')
print(f'Max units sold:  {units_nonzero.max():,}')

# --- Zero-streak analysis ---
print(f'\n{"═"*55}')
print('ZERO-STREAK ANALYSIS')
print(f'{"═"*55}')

def max_zero_streak(s):
    """Return the longest consecutive run of zeros in series s."""
    max_streak = streak = 0
    for v in s:
        if v == 0:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak

# Sample 2,000 series for speed — full 30k would take several minutes
sample_ids = df['id'].unique()[:2000]
streaks = (
    df[df['id'].isin(sample_ids)]
    .sort_values(['id', 'date'])
    .groupby('id', observed=True)['units_sold']
    .apply(max_zero_streak)
)

print(f'\nMax consecutive zero-sales days per series (sample n=2,000):')
print(streaks.describe().round(1))
print(f'\n% series with max streak > 30 days:  {(streaks > 30).mean()*100:.1f}%')
print(f'% series with max streak > 90 days:  {(streaks > 90).mean()*100:.1f}%')
print(f'% series with max streak > 180 days: {(streaks > 180).mean()*100:.1f}%')
print(f'% series with max streak > 365 days: {(streaks > 365).mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(streaks, bins=60, color='steelblue', edgecolor='none')
ax.axvline(30,  color='orange', linestyle='--', linewidth=1.5, label='30 days')
ax.axvline(180, color='red',    linestyle='--', linewidth=1.5, label='180 days')
ax.set_title('Distribution of Longest Zero-Sales Streak per Series', fontsize=13)
ax.set_xlabel('Max Consecutive Zero-Sales Days')
ax.set_ylabel('Number of Series')
ax.legend()
plt.tight_layout()
plt.show()

### Units Sold Distribution & Zero-Streak Results

All metrics below are measured at the **product-store-day** level — one
specific product at one specific store on one specific day.

**Units sold distribution (non-zero days):**
- Median of 2 units, 75th percentile of 4 units, max of 763. On a typical
  day where a specific product actually sold something at a specific store,
  it sold 2 units — not 2 units across all stores or all products.
- Skewness of 11.89 — heavily right-skewed. 91.8% of non-zero days sell
  3 or fewer units per product per store; large volume days are rare but
  extreme (max 763 units in a single day at one store).
- Individual series are sparse and lumpy, not smooth continuous processes.
  At the **aggregate level** the law of large numbers smooths this
  considerably — monthly aggregate revenue is well-behaved for SARIMA.
  At the **individual product-store level**, forecast intervals will be
  wide and should be interpreted conservatively.

**Zero-streak analysis:**
- Mean longest zero streak per product-store series: **430 consecutive days**.
  Median: **257 days**. 44.1% of series have a streak exceeding 365 days —
  meaning one specific product at one specific store recorded zero sales
  every single day for over a year at some point in its history.
- These are not random intermittent demand zeros — they are **structural
  zeros** reflecting real product lifecycle events: products not yet stocked,
  discontinued, or delisted from a specific store. A product absent from
  shelves for 400 days has zero sales because it doesn't exist there, not
  because demand happened to be zero.
- This reframes the 68.2% zero rate from Section 5 and the 8.1% complete
  series rate from Section 15 — both are driven primarily by product
  lifecycle gaps at the store level, not sparse demand patterns.

**Modeling implications:**
- **SARIMA:** Only viable on the 2,469 complete series. Structural zero
  gaps in incomplete series break the continuous time index assumption.
- **XGBoost:** Lag features (`lag_7`, `lag_28`) engineered across a
  structural zero gap are meaningless — lagging back into a period where
  the product didn't exist at that store produces misleading signal. In
  notebook 3, lag features must only be computed within active selling
  windows, not across zero gaps.
- **Anomaly detection:** A sudden return to sales after a long zero streak
  is a product relaunch at that store, not a demand spike. The anomaly
  detection in notebook 5 must account for this or it will generate false
  positives on every product introduction.

## 23. Key Findings & EDA Summary

### Dataset & Quality
- **30,490 product-store time series** across 3,049 products, 10 stores,
  3 states (CA, TX, WI) from Jan 29 2011 – Apr 24 2016 (64 months)
- Raw data comes in wide format (1,913 day columns) — reshaped to long
  format before any analysis. Full frame is 58M rows and held in memory
  only during EDA; downstream notebooks rebuild it from raw files directly.
- Revenue computed as `units_sold × sell_price` on demand via lightweight
  aggregated joins — never materialized at full 58M row scale.
- **68.2% of product-store-days have zero sales** — measured at the granularity
  of one specific product at one specific store on one specific day.

### Units Sold & Zero Structure
- **Median of 2 units per product-store-day** on non-zero days. 91.8% of
  non-zero days sell 3 or fewer units. Skewness of 11.89 — heavily right-skewed
  with rare but extreme spikes (max 763 units at one store in one day).
- **Zero-inflation is structural, not random.** Mean longest zero streak per
  product-store series is 430 consecutive days; 44.1% of series have a streak
  exceeding 365 days. These are product lifecycle gaps — a product not yet
  stocked, discontinued, or delisted at a specific store — not intermittent demand.
- This reframes the 68.2% zero rate and the 8.1% complete series rate — both
  are driven by product lifecycle events, not sparse demand patterns.

### Revenue & Category Structure
- **FOODS dominates at 58%** of total revenue, HOUSEHOLD 29.8%, HOBBIES 12.2%
- **FOODS_3 alone is 37.8%** of all revenue — the single most important department
- **CA_3 is the highest revenue store at 17.1%** — nearly 3x the smallest store
  (CA_4 at 6.5%). Store size varies significantly; store-level encoding is essential.
- **HOBBIES_2 is negligible at 0.63%** — too sparse for reliable individual modeling

### Trend, Seasonality & Stationarity
- **Strong upward trend** — aggregate revenue roughly doubled from ~$2M/month
  in 2011 to ~$4M/month by 2016. Trend dominates over seasonality.
- **Seasonal effects are modest** — August (+$147K) and March (+$128K) are the
  strongest months. December is slightly negative — stores close on Christmas day.
- **Departments are highly correlated (0.64–0.96)** but not perfectly —
  department-level features add genuine signal beyond the aggregate trend.
- **ADF test results:**
  - Aggregate series is non-stationary raw (p=0.318) — seasonal differencing
    resolves it cleanly (p≈0.000). SARIMA order: d=0, D=1.
  - Representative series is already stationary raw (p=0.0004). SARIMA order: d=0, D=0.
- **ACF/PACF parameter decisions:**
  - Aggregate candidate: SARIMA(1,0,0)(0,1,0)[12]
  - Representative candidate: SARIMA(1,0,1)(0,0,0)[12]
  - Both confirmed via grid search in notebook 2 with AIC selection.

### Event & SNAP Effects
- **Labor Day (+19.6%) and Super Bowl (+18.9%)** are the strongest positive
  event drivers — pre-gathering stock-up behavior.
- **Christmas (-100%) and Thanksgiving (-40.8%)** — stores close or traffic
  shifts to surrounding days. These need lead/lag features, not just binary flags.
- **SNAP effect is strong and state-specific** — WI FOODS shows +32.5% uplift
  on SNAP days, TX FOODS +17.2%, CA FOODS +10.3%. State-level SNAP flags
  are among the most valuable XGBoost features.

### Price Volatility & Elasticity
- **All departments show cv% above 50%** — price changes are universal,
  not isolated to one category. HOBBIES_1 most volatile (83.1% cv),
  FOODS departments most stable (62–64% cv).
- **Price elasticity is product-specific, not department-level.** Department
  aggregate correlations are near zero (r = 0.003–0.129) — price changes on
  one product are diluted across thousands of others when aggregated.
- **At the individual product-store level, asymmetric elasticity is confirmed:**
  price drops produce a strong demand response (r=0.553, ~77% demand increase
  per 10% price drop for FOODS_3_383 @ CA_3) while price increases show a
  much weaker effect (r=0.180). Customers stock up on discounts but do not
  proportionally reduce purchases when prices rise.
- **Confounding at aggregate levels:** Walmart discounts slow-moving products,
  so price drops and low demand co-occur — producing a spurious positive slope
  at the department level. This effect disappears at the product-store level.
- **Feature engineering decisions:** `sell_price`, `price_change_pct` (signed),
  `price_drop_pct`, and `price_increase_pct` must all be computed at the
  `item_id` + `store_id` level. Department-average price features are excluded.

### Series Completeness
- **Only 2,469 of 30,490 series (8.1%) have complete 64-month histories** —
  individual SARIMA is only viable for these. The incompleteness is driven by
  product lifecycle gaps, not data quality issues.
- **XGBoost handles incomplete series** via row-level feature engineering but
  lag features must not be computed across structural zero gaps.
- Representative series: **FOODS_3_163_CA_3_validation** — median revenue
  complete series from the highest-revenue department and store.

### Modeling Decisions Locked In

| Decision | Justification |
|---|---|
| SARIMA with period=12 | Seasonal decomposition and ACF/PACF confirm annual cycle |
| Aggregate: d=0, D=1 | ADF confirms seasonal differencing resolves non-stationarity |
| Representative: d=0, D=0 | ADF confirms already stationary raw |
| Aggregate SARIMA candidate: (1,0,0)(0,1,0)[12] | PACF cuts off at lag-1; no seasonal spikes at lag-12 |
| Representative SARIMA candidate: (1,0,1)(0,0,0)[12] | PACF lag-1 only; ACF cuts off at lag-2 |
| Additive decomposition | Seasonal amplitude does not grow with trend level |
| Aggregate + representative series | Both modeled separately, different variance profiles |
| XGBoost features: SNAP, holidays, price | All quantified and confirmed significant in EDA |
| Christmas/Thanksgiving as lead features | Day-of effect is negative — signal lives in surrounding days |
| Store + department encoding required | CA_3 is 3x CA_4; departments have independent variation |
| Price features at product-store level only | Department-level price correlations r < 0.13 — no signal |
| Signed price change features required | Asymmetric elasticity confirmed — drops and increases differ |
| Lag features within active windows only | Structural zero gaps make cross-gap lags meaningless |
| Anomaly detection must filter product launches | Return from long zero streak is relaunch, not demand spike |
| Walk-forward CV | 5+ years of history enables multiple meaningful folds |
| Log transform — defer | Assess monthly aggregate variance in SARIMA notebook |

### Limitations
- Revenue is derived (`units × price`), not directly observed — price join
  introduces minor gaps for products not yet stocked
- 64 months of aggregate data is sufficient but not abundant for SARIMA —
  interpret confidence intervals conservatively
- Individual product-store series are sparse and noisy (median 2 units/day,
  skewness 11.89) — individual SARIMA forecasts carry wide uncertainty intervals
- Full 58M row dataframe cannot be saved to CSV without hitting memory and
  storage limits — downstream notebooks rebuild from raw files
- Zero-streak analysis based on a sample of 2,000 series — directionally
  accurate but exact percentages may shift slightly across the full 30,490
- Price elasticity coefficients at the single product level (n=16 drops,
  n=13 increases) carry wide uncertainty — directionally valid but not
  precise estimates

## 24. Column Reference

Complete reference for every column in `df` — the primary in-memory dataframe
built during EDA and reconstructed from raw files in modeling notebooks.
Note: `sell_price` and `revenue` are not columns in `df` — they are joined
on demand in specific sections to avoid memory overflow.

All observations in `df` are at the **product-store-day** granularity:
one row = one specific product, at one specific store, on one specific day.

### Identity & Hierarchy

| Column | Type | Description |
|---|---|---|
| `id` | category | Unique product-store identifier (e.g. `FOODS_1_001_CA_1_validation`) |
| `item_id` | category | Product identifier (e.g. `FOODS_1_001`) |
| `dept_id` | category | Department (e.g. `FOODS_1`, `HOBBIES_2`) — 7 unique |
| `cat_id` | category | Category: `FOODS`, `HOBBIES`, or `HOUSEHOLD` |
| `store_id` | category | Store identifier (e.g. `CA_1`, `TX_2`) — 10 unique |
| `state_id` | category | State: `CA`, `TX`, or `WI` |

### Time

| Column | Type | Description |
|---|---|---|
| `date` | datetime | Calendar date of the observation |
| `d` | object | M5 day ID (e.g. `d_1` through `d_1913`) — join key to calendar |
| `wm_yr_wk` | int64 | Walmart week ID — join key for `sell_prices.csv` |
| `weekday` | category | Day of week (e.g. `Monday`) |
| `month` | int64 | Month number (1–12) |
| `year` | int64 | Calendar year |
| `month_dt` | datetime | First day of the month — used for all monthly aggregations |

### Events & SNAP

| Column | Type | Description |
|---|---|---|
| `event_name_1` | category (nullable) | Primary holiday or event name (e.g. `SuperBowl`, `Thanksgiving`) |
| `event_type_1` | category (nullable) | Event category: `National`, `Religious`, `Cultural`, or `Sporting` |
| `snap_CA` | int64 (0/1) | Whether CA residents received SNAP benefits this day |
| `snap_TX` | int64 (0/1) | Whether TX residents received SNAP benefits this day |
| `snap_WI` | int64 (0/1) | Whether WI residents received SNAP benefits this day |

### Sales

| Column | Type | Description |
|---|---|---|
| `units_sold` | int16 | Units of this specific product sold at this specific store on this specific day |

### Joined On Demand (not in df)

| Column | Source | Description |
|---|---|---|
| `sell_price` | `sell_prices.csv` | Weekly sell price in USD — joined via `store_id`, `item_id`, `wm_yr_wk` |
| `revenue` | derived | `units_sold × sell_price` — computed after price join |

---

### Processed Files Saved

| File | Rows | Used In |
|---|---|---|
| `monthly_aggregate.csv` | 64 | SARIMA, Prophet — aggregate series |
| `monthly_series_FOODS_3_163_CA_3_validation.csv` | 64 | SARIMA, Prophet — individual series |
| `complete_series_ids.csv` | 2,469 | Notebook 3 — filter to modelable series |
| `sell_prices.csv` (raw) | 6.8M | Notebook 3 — joined during feature engineering |

### Hardcoded Values for All Modeling Notebooks

| Parameter | Value |
|---|---|
| Representative series ID | `FOODS_3_163_CA_3_validation` |
| Seasonal period | `12` |
| Train months | 48 |
| Test months | 15  |
| Forecast horizon (SARIMA/Prophet) | 12 months |
| Forecast horizon (XGBoost) | 28 days |